FED — Create Summary Table

In [1]:
import pandas as pd
import numpy as np
import os
import re
from datetime import time
from pathlib import Path

csv_pattern = re.compile(r'^(?!.*Periodic).*\.csv$', re.IGNORECASE)

# PARAMÈTRES
folder_path = "//10.69.168.1/crnldata/forgetting/Carla/Fed"
debut_poke  = 7    # Premier poke du bloc de début
taille_bloc = 100  # Nombre de pokes par bloc
output_folder = os.path.join(folder_path, "FED_Summary_table.xlsx")


#  FONCTIONS
specific_events_list = [
    'LeftDuringDispense', 'RightDuringDispense',
    'LeftWithPellet',     'RightWithPellet',
    'RightinTimeOut',     'LeftinTimeOut'
]

def get_period(h):
    if pd.isnull(h):
        return None
    return "Day" if time(7, 0, 1) <= h <= time(19, 0, 0) else "Night"


def compute_impulsivite(block_df):
    """
    Calcule le nombre d'erreurs avant le premier poke correct après chaque pellet.
    Retourne mean/median avec et sans les 0.
    """
    scores = []
    en_attente = False
    compteur   = 0

    for _, row in block_df.iterrows():
        event      = str(row['Event']).strip()
        active     = str(row['Active_Poke']).strip()

        if event == 'Pellet':
            compteur   = 0
            en_attente = True
            continue

        if en_attente:
            if event in ['Left', 'Right']:
                if event == active:
                    scores.append(compteur)
                    compteur   = 0
                    en_attente = False
                else:
                    compteur += 1

    if len(scores) == 0:
        return {
            'Mean_Errors_before_Correct':      np.nan,
            'Median_Errors_before_Correct':    np.nan,
            
        }

    s      = np.array(scores)
    s_no0  = s[s > 0]

    return {

        'Mean_Errors_before_Correct':   round(s_no0.mean(),   2) if len(s_no0) > 0 else np.nan,
        'Median_Errors_before_Correct': round(float(np.median(s_no0)), 2) if len(s_no0) > 0 else np.nan,
    }


def compute_params(block_df):
    """Calcule tous les paramètres pour un bloc de données."""

    left_pokes  = (block_df['Event'] == 'Left').sum()
    right_pokes = (block_df['Event'] == 'Right').sum()
    total_pokes = left_pokes + right_pokes
    pellets     = (block_df['Event'] == 'Pellet').sum()

    poke_rows   = block_df[block_df['Event'].isin(['Left', 'Right'])]
    correct     = (poke_rows['Event'] == poke_rows['Active_Poke']).sum()
    pct_correct = round(correct / total_pokes * 100, 2) if total_pokes > 0 else np.nan

    # Premier poke après chaque pellet (Left/Right uniquement)
    pellet_positions = block_df[block_df['Event'] == 'Pellet'].index.tolist()
    first_pokes = []
    for idx in pellet_positions:
        loc = block_df.index.get_loc(idx)
        remaining = block_df.iloc[loc + 1:]
        poke_rows = remaining[remaining['Event'].isin(['Left', 'Right'])]
        if not poke_rows.empty:
            first_pokes.append(poke_rows.iloc[0])

            
    fp_df      = pd.DataFrame(first_pokes)
    fp_total   = len(fp_df)
    fp_correct = int((fp_df['Event'] == fp_df['Active_Poke']).sum()) if not fp_df.empty else 0
    pct_fp     = round(fp_correct / fp_total * 100, 2) if fp_total > 0 else np.nan

    # Retrieval Time
    rt        = block_df[block_df['Event'] == 'Pellet']['Retrieval_Time'].dropna()
    mean_rt   = round(rt.mean(),   3) if len(rt) > 0 else np.nan
    median_rt = round(rt.median(), 3) if len(rt) > 0 else np.nan

    # Inter-Pellet Interval
    ipi        = block_df[block_df['Event'] == 'Pellet']['InterPelletInterval'].dropna()
    mean_ipi   = round(ipi.mean(),   2) if len(ipi) > 0 else np.nan
    median_ipi = round(ipi.median(), 2) if len(ipi) > 0 else np.nan

    # Durée
    start_dt = block_df['Datetime'].min()
    end_dt   = block_df['Datetime'].max()
    dur_min  = round((end_dt - start_dt).total_seconds() / 60, 1) \
               if pd.notna(start_dt) and pd.notna(end_dt) else np.nan

    spec = block_df[block_df['Event'].isin(specific_events_list)].shape[0]

    # Impulsivité
    impuls = compute_impulsivite(block_df)

    return {
        'Pellets':                          int(pellets),
        'Total Pokes':                      int(total_pokes),
        'Left Pokes':                       int(left_pokes),
        'Right Pokes':                      int(right_pokes),
        'Correct Pokes':                    int(correct),
        '% Correct':                        pct_correct,
        'First Poke':                       fp_total,
        'First Poke Correct':               fp_correct,
        '% First Poke Correct':             pct_fp,
        'Mean Retrieval Time s':            mean_rt,
        'Specific Events':                  int(spec),
        **impuls
    }


def analyse_csv(filepath):
    """
    Charge un CSV FED et retourne une liste de dicts :
      - une ligne par Period Block (Day1, Night1, Day2...)
      - une ligne 100_Debut (pokes 7-106)
      - une ligne 100_Fin   (100 derniers pokes)
    """
    parts    = Path(filepath).parts
    mois     = parts[-2]
    souris   = parts[-3]
    genotype = parts[-4]

    try:
        with open(filepath, 'r') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        df = pd.read_csv(filepath, sep=sep, na_values=['nan'])
        df.columns = df.columns.str.replace(r'\+AF8-', '_', regex=True)
        df.columns = df.columns.str.strip()
        for col in ['Retrieval_Time', 'InterPelletInterval', 'Poke_Time', 'Motor_Turns']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        # Vérification colonnes obligatoires
        required = ['Event', 'Active_Poke', 'Retrieval_Time', 'InterPelletInterval']
        missing  = [c for c in required if c not in df.columns]
        if missing:
            print(f"    Colonnes manquantes {missing} dans : {filepath}")
            return []
    except Exception as e:
        print(f"    Erreur lecture {filepath} : {e}")
        return []

    # Préparation
    df['Datetime'] = pd.to_datetime(df['MM:DD:YYYY hh:mm:ss'], errors='coerce')
    df['Date']     = df['Datetime'].dt.date
    df['_Hour']    = df['Datetime'].dt.time
    df['Period']   = df['_Hour'].apply(get_period)
    df = df.sort_values('Datetime').reset_index(drop=True)

    base_info = {
        'Souris':   souris,
        'Genotype': genotype,
        'Mois':     mois,
        'Fichier':  os.path.basename(filepath),
    }

    rows = []
 

    #  PARTIE 1 : lignes Day/Night 
    
    df['Period_Block'] = (df['Period'] != df['Period'].shift(1)).cumsum()
    period_counter     = {'Day': 0, 'Night': 0}

    for period_block, group in df.groupby('Period_Block', sort=True):
        period_type = group['Period'].iloc[0]
        if period_type not in ['Day', 'Night']:
            continue

        period_counter[period_type] += 1
        period_label = f"{period_type}{period_counter[period_type]}"

        params = compute_params(group)

        rows.append({
            **base_info,
            'Row_Type':     'Period_Block',
            'Period_Label': period_label,
            'Pokes_Range':  '',
            'Start_Date':   str(group['Date'].min()),
            'End_Date':     str(group['Date'].max()),
            **params
        })


    #  PARTIE 2 : blocs 100 pokes  
    
    poke_mask = df['Event'].isin(['Left', 'Right'])
    df['Poke_Number'] = np.nan
    df.loc[poke_mask, 'Poke_Number'] = range(1, poke_mask.sum() + 1)
    total_pokes_count = poke_mask.sum()

    fin_poke        = debut_poke + taille_bloc - 1
    last_poke_start = total_pokes_count - taille_bloc + 1

    blocs_100 = [
        ("100_Debut", debut_poke, fin_poke),
        ("100_Fin",   last_poke_start, total_pokes_count),
    ]

    for bloc_name, p_start, p_end in blocs_100:
        if p_start < 1 or p_end > total_pokes_count or p_start > p_end:
            continue

        pokes_b = df[poke_mask & (df['Poke_Number'] >= p_start) & (df['Poke_Number'] <= p_end)]
        if pokes_b.empty:
            continue

        idx_s   = pokes_b.index.min()
        idx_e   = pokes_b.index.max()
        bloc_df = df.loc[idx_s:idx_e]

        params = compute_params(bloc_df)

        rows.append({
            **base_info,
            'Row_Type':     'Pokes_100',
            'Period_Label': bloc_name,
            'Pokes_Range':  f"{p_start}-{p_end}",
            'Start_Date':   str(bloc_df['Date'].min()),
            'End_Date':     str(bloc_df['Date'].max()),
            **params
        })

    return rows


print("Fonctions définies")





Fonctions définies


In [2]:
#   PARCOURS DE TOUS LES FICHIERS CSV
all_rows    = []
csv_count   = 0
error_count = 0

for root, dirs, files in os.walk(folder_path):
    
    csv_files = [f for f in files if csv_pattern.match(f)]

    for csv_file in csv_files:
        filepath = os.path.join(root, csv_file)

        rel   = os.path.relpath(filepath, folder_path)
        depth = len(Path(rel).parts)
        if depth != 4:
            continue

        print(f" {rel}")
        rows = analyse_csv(filepath)

        if rows:
            all_rows.extend(rows)
            csv_count += 1
        else:
            error_count += 1

print('OK')

 APPPS1\AHAD01.37\10 mois\AHAD01.37_27_10_2025au29_10_2025.CSV
 APPPS1\AHAD01.37\11 mois\AHAD01.37_01_12_2025au03_12_2025.CSV
 APPPS1\AHAD01.37\11 mois\AHAD01.37_01_12_2025au03_12_2025_100pokes_analysis.csv
    Colonnes manquantes ['Event', 'Active_Poke', 'Retrieval_Time', 'InterPelletInterval'] dans : //10.69.168.1/crnldata/forgetting/Carla/Fed\APPPS1\AHAD01.37\11 mois\AHAD01.37_01_12_2025au03_12_2025_100pokes_analysis.csv
 APPPS1\AHAD01.37\11 mois\AHAD01.37_01_12_2025au03_12_2025_tous_blocs_100pokes.csv
    Colonnes manquantes ['Event', 'Active_Poke', 'Retrieval_Time', 'InterPelletInterval'] dans : //10.69.168.1/crnldata/forgetting/Carla/Fed\APPPS1\AHAD01.37\11 mois\AHAD01.37_01_12_2025au03_12_2025_tous_blocs_100pokes.csv
 APPPS1\AHAD01.37\12 mois\AHAD01.37_05_01_2026au07_01_2026.CSV
 APPPS1\AHAD01.37\3 mois\AHAD01.37_27_03_2025au04_04_2025.CSV
 APPPS1\AHAD01.37\3 mois\AHAD01.37_27_03_2025au04_04_2025_impulsivite.csv
    Colonnes manquantes ['Event', 'Active_Poke', 'Retrieval_Time', 

In [3]:
#  TABLEAU FINAL

Summary_table = pd.DataFrame(all_rows)

# Trier : Genotype > Souris > Mois > Period_Block d'abord, puis Pokes_100
Summary_table['_sort_type'] = Summary_table['Row_Type'].map({'Period_Block': 0, 'Pokes_100': 1})
Summary_table = Summary_table.sort_values(
    ['Souris', 'Genotype', 'Mois', '_sort_type']
).drop(columns=['_sort_type']).reset_index(drop=True)

display(Summary_table.head(12))


#  SAUVEGARDE 

with pd.ExcelWriter(output_folder, engine='openpyxl') as writer:
    Summary_table.to_excel(writer, index=False, sheet_name='FED_Summary')

print(f" {output_folder}")
print(f" {Summary_table['Souris'].nunique()} souris")

,Souris,Genotype,Mois,Fichier,Row_Type,Period_Label,Pokes_Range,Start_Date,End_Date,Pellets,...,Right Pokes,Correct Pokes,% Correct,First Poke,First Poke Correct,% First Poke Correct,Mean Retrieval Time s,Specific Events,Mean_Errors_before_Correct,Median_Errors_before_Correct
0,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Period_Block,Day1,,2025-10-27,2025-10-27,9,...,8,9,28.12,8,4,50.00,3.346,2,5.75,5.0
1,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Period_Block,Night1,,2025-10-27,2025-10-28,131,...,214,131,31.12,130,73,56.15,2.600,7,4.93,4.0
2,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Period_Block,Day2,,2025-10-28,2025-10-28,32,...,96,32,22.54,32,15,46.88,3.057,4,6.62,8.0
3,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Period_Block,Night2,,2025-10-28,2025-10-29,110,...,89,110,68.32,109,81,74.31,2.520,14,1.78,1.0
4,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Period_Block,Day3,,2025-10-29,2025-10-29,12,...,12,12,44.44,11,6,54.55,0.408,0,3.00,3.0
5,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Pokes_100,100_Debut,7-106,2025-10-27,2025-10-27,21,...,21,21,21.00,21,9,42.86,2.020,2,6.27,5.0
6,AHAD01.37,APPPS1,10 mois,AHAD01.37_27_10_2025au29_10_2025.CSV,Pokes_100,100_Fin,684-783,2025-10-29,2025-10-29,60,...,58,61,61.00,60,41,68.33,1.381,2,2.05,2.0
7,AHAD01.37,APPPS1,11 mois,AHAD01.37_01_12_2025au03_12_2025.CSV,Period_Block,Day1,,2025-12-01,2025-12-01,22,...,78,22,23.40,22,10,45.45,0.063,0,6.36,3.0
8,AHAD01.37,APPPS1,11 mois,AHAD01.37_01_12_2025au03_12_2025.CSV,Period_Block,Night1,,2025-12-01,2025-12-02,132,...,170,132,45.99,131,94,71.76,0.269,10,3.06,2.0
9,AHAD01.37,APPPS1,11 mois,AHAD01.37_01_12_2025au03_12_2025.CSV,Period_Block,Day2,,2025-12-02,2025-12-02,2,...,15,2,12.50,2,1,50.00,0.090,1,NaN,NaN


 //10.69.168.1/crnldata/forgetting/Carla/Fed\FED_Summary_table.xlsx
 41 souris
